In [ ]:
 
#For removing the transitive dependencies

# RUN THIS CELL TO SETUP THE CHALLENGE DATA

import pandas as pd

import sqlite3

conn_challenge = sqlite3.connect(':memory:')

challenge_data = {

"Visit_ID": [5001, 5001, 5002, 5003],

"Student_ID": [101, 101, 102, 104],

"Student_Name": ["Alice", "Alice", "Bob", "David"],

"Doctor_ID": ["DOC_XYZ", "DOC_XYZ", "DOC_ABC", "DOC_XYZ"],

"Doctor_Name": ["Dr. Evans", "Dr. Evans", "Dr. Green", "Dr. Evans"],

"Doctor_Clinic": ["General Medicine", "General Medicine", "Sports Med", "General Medicine"],

"Prescriptions": ["Amoxicillin, Ibuprofen", "Amoxicillin, Ibuprofen", "Bandages", "Vitamin D"]

}
df_challenge = pd.DataFrame(challenge_data)

df_challenge.to_sql('Patient_Visits_ONF', conn_challenge, index=False, if_exists='replace')

print("--- Challenge Dataset (ONF) ---")

print(df_challenge)

--- Challenge Dataset (ONF) ---
   Visit_ID  Student_ID Student_Name Doctor_ID Doctor_Name     Doctor_Clinic  \
0      5001         101        Alice   DOC_XYZ   Dr. Evans  General Medicine   
1      5001         101        Alice   DOC_XYZ   Dr. Evans  General Medicine   
2      5002         102          Bob   DOC_ABC   Dr. Green        Sports Med   
3      5003         104        David   DOC_XYZ   Dr. Evans  General Medicine   

            Prescriptions  
0  Amoxicillin, Ibuprofen  
1  Amoxicillin, Ibuprofen  
2                Bandages  
3               Vitamin D  


In [4]:
#1. Why does this table violate 1NF? Which column is the culprit?
print("""The table violates 1NF because the Prescriptions column contains multiple values in a single cell.

Example: "Amoxicillin, Ibuprofen"    
A 1NF table requires each cell to contain only one atomic value.

Culprit Column: Prescriptions""")

The table violates 1NF because the Prescriptions column contains multiple values in a single cell.

Example: "Amoxicillin, Ibuprofen"    
A 1NF table requires each cell to contain only one atomic value.

Culprit Column: Prescriptions


In [9]:
#2. Assume Visit_ID is the primary key. Why do Student_Name and Doctor_Clinic violate 2NF and 3NF? 

print("""Since Visit_ID is a single-column primary key, the table automatically satisfies 2NF. 
However, Student_Name depends on Student_ID and Doctor_Clinic depends on Doctor_ID rather than directly on Visit_ID. 
These are transitive dependencies (Visit_ID → Student_ID → Student_Name and Visit_ID → Doctor_ID → Doctor_Clinic), 
which violate 3NF""")

Since Visit_ID is a single-column primary key, the table automatically satisfies 2NF. 
However, Student_Name depends on Student_ID and Doctor_Clinic depends on Doctor_ID rather than directly on Visit_ID. 
These are transitive dependencies (Visit_ID → Student_ID → Student_Name and Visit_ID → Doctor_ID → Doctor_Clinic), 
which violate 3NF


In [7]:
# 3. If knowing Doctor_ID immediately tells you Doctor_Clinic,
# but Doctor_ID is not a primary key, which normal form is violated?
print("""
     Doctor_ID determines Doctor_Clinic:
     Doctor_ID → Doctor_Clinic
     Since Doctor_ID is not the primary key, Doctor_Clinic depends on a non-key attribute 
     rather than directly on the primary key (Visit_ID). 
     This creates a transitive dependency:
     Visit_ID → Doctor_ID → Doctor_Clinic
     Therefore, Third Normal Form (3NF) is violated. 
     To achieve 3NF, Doctor_ID, Doctor_Name, and Doctor_Clinic should be moved to a separate Doctor table""") 


     Doctor_ID determines Doctor_Clinic:
     Doctor_ID → Doctor_Clinic
     Since Doctor_ID is not the primary key, Doctor_Clinic depends on a non-key attribute 
     rather than directly on the primary key (Visit_ID). 
     This creates a transitive dependency:
     Visit_ID → Doctor_ID → Doctor_Clinic
     Therefore, Third Normal Form (3NF) is violated. 
     To achieve 3NF, Doctor_ID, Doctor_Name, and Doctor_Clinic should be moved to a separate Doctor table


In [8]:
# ============================================================
# TASK 2 : Fix with Code (Convert to 1NF)
# ============================================================

# Split multi-valued prescriptions into atomic values

df_task_1nf = df_challenge.assign(
    Prescriptions=df_challenge['Prescriptions'].str.split(',')
).explode('Prescriptions')

# Remove extra spaces
df_task_1nf['Prescriptions'] = (
    df_task_1nf['Prescriptions'].str.strip()
)

# Save normalized table to SQL
df_task_1nf.to_sql(
    'Patient_Visits_1NF',
    conn_challenge,
    index=False,
    if_exists='replace'
)

print("\n--- Patient Visits in 1NF ---")

print(
    f"Row count increased from "
    f"{len(df_challenge)} to {len(df_task_1nf)}"
)

# Display normalized dataframe
display(df_task_1nf)


--- Patient Visits in 1NF ---
Row count increased from 4 to 6


,Visit_ID,Student_ID,Student_Name,Doctor_ID,Doctor_Name,Doctor_Clinic,Prescriptions
0,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Amoxicillin
0,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Ibuprofen
1,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Amoxicillin
1,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Ibuprofen
2,5002,102,Bob,DOC_ABC,Dr. Green,Sports Med,Bandages
3,5003,104,David,DOC_XYZ,Dr. Evans,General Medicine,Vitamin D
